In [1]:
import numpy as np
import pandas as pd
import torch
import re
from pathlib import Path

In [2]:
def _to_1d_array(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple, np.ndarray)):
        arr = np.array(x)
    else:
        return None
    if arr.shape == ():
        return None
    return arr.reshape(-1)

def extract_dice_and_count(value):
    arr = _to_1d_array(value)
    if arr is None or arr.size == 0:
        return None, None
    return float(np.nanmean(arr)), int(arr.size)

In [3]:
RESULT_DIR = Path('../../results/unet_eval_auto').resolve()
pt_files = sorted([p for p in RESULT_DIR.glob('*.pt')])
pattern_dataset = re.compile(r'^(?P<root>pmri|mnmv2)')

# Rebuild results with dice + ece
row_records = []
for f in pt_files:
    data = torch.load(f, map_location='cpu')
    for split_key, metrics in data.items():
        if not isinstance(metrics, dict):
            continue
        if 'dice' not in metrics:
            continue
        dice_val, case_count = extract_dice_and_count(metrics['dice'])
        if dice_val is None:
            continue
        ece_val, _ = extract_dice_and_count(metrics.get('ece', None))
        dataset_guess = pattern_dataset.match(f.name).group('root') if pattern_dataset.match(f.name) else 'unknown'
        context = f.stem
        row_records.append({
            'file': f.name,
            'dataset': dataset_guess,
            'context': context,
            'split': split_key.split('_')[-1],
            'dice': dice_val,
            'ece': ece_val,
            'case_count': case_count,
        })

results_df = pd.DataFrame(row_records)
print('Collected rows:', len(results_df))
print('Columns found:', results_df.columns.tolist())
if 'dice' in results_df.columns:
    coverage = results_df['dice'].notna().mean()
    print(f'Dice coverage: {coverage*100:.1f}% of rows')
if 'ece' in results_df.columns:
    coverage = results_df['ece'].notna().mean()
    print(f'ECE coverage: {coverage*100:.1f}% of rows')

if not results_df.empty:
    dice_overview = results_df.pivot_table(index='context', columns='split', values='dice')
    dice_overview = dice_overview.sort_values(by=[c for c in ['test','val','train'] if c in dice_overview.columns], ascending=False)
    print('\nDice overview sample:')
    print(dice_overview.head())
else:
    dice_overview = pd.DataFrame()

results_df

Collected rows: 9
Columns found: ['file', 'dataset', 'context', 'split', 'dice', 'ece', 'case_count']
Dice coverage: 100.0% of rows
ECE coverage: 100.0% of rows

Dice overview sample:
split                             test     train       val
context                                                   
pmri_promise12                0.668256  0.935265  0.866527
pmri_threet-to-onepointfivet  0.666087  0.928823  0.854488
pmri_onepointfivet-to-threet  0.517944  0.954817  0.863758


/tmp/ipykernel_4027448/1645493342.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(f, map_location='cpu')


,file,dataset,context,split,dice,ece,case_count
0,pmri_onepointfivet-to-threet.pt,pmri,pmri_onepointfivet-to-threet,train,0.954817,0.161869,432
1,pmri_onepointfivet-to-threet.pt,pmri,pmri_onepointfivet-to-threet,val,0.863758,0.228395,49
2,pmri_onepointfivet-to-threet.pt,pmri,pmri_onepointfivet-to-threet,test,0.517944,0.285054,1121
3,pmri_promise12.pt,pmri,pmri_promise12,train,0.935265,0.189810,461
4,pmri_promise12.pt,pmri,pmri_promise12,val,0.866527,0.207210,64
5,pmri_promise12.pt,pmri,pmri_promise12,test,0.668256,0.259523,1248
6,pmri_threet-to-onepointfivet.pt,pmri,pmri_threet-to-onepointfivet,train,0.928823,0.175536,1018
7,pmri_threet-to-onepointfivet.pt,pmri,pmri_threet-to-onepointfivet,val,0.854488,0.229805,103
8,pmri_threet-to-onepointfivet.pt,pmri,pmri_threet-to-onepointfivet,test,0.666087,0.267051,481


# METRICS TABLE

$$
\begin{array}{lccc}
\hline
\mathbf{Dataset} & \mathbf{Training} & \mathbf{Validation} & \mathbf{Shift} \\
\hline
\text{M\&M Shifts} & & & \\
\text{  Scanner} & .913\ (2390) & .870\ (252) & .864\ (3470) \\
\text{  Pathology} & .928\ (1085) & .906\ (128) & .724\ (587) \\
\\
\text{PMRI Shifts} & & & \\
\text{  Dataset} & .927\ (461) & .878\ (64) & .615\ (1248) \\
\text{  3T to 1.5T} & .927\ (1018) & .862\ (103) & .716\ (481) \\
\\
\text{MY PMRI Shifts} & & & \\
\text{  Dataset} & .935\ (461) & .867\ (64) & .668\ (1248) \\
\text{  3T to 1.5T} & .929\ (1018) & .854\ (103) & .666\ (481) \\
\text{  1.5T to 3T} & .955\ (432) & .864\ (49) & .518\ (1121) \\
\hline
\end{array}
$$

In [ ]:
metrics